# 03 · ResNet-18 Fine-Tuning – Reference Baseline (Hannah)

ImageNet-pretrained ResNet-18 (He et al., 2016) fine-tuned on GTSRB.

**Two-phase fine-tuning**
1. *Head only* – freeze the backbone, train the new 43-class head (fast, avoids wrecking pretrained features).
2. *Full network* – unfreeze everything and train with a smaller learning rate.

Design notes
- Normalisation with ImageNet mean/std happens **inside** the model, so data loaders just need to give images in [0, 1] (same input as every other model, including later corrupted images).
- Images are bilinearly upsampled to `resize_to` inside the model, because ResNet downsamples by 32× and 48×48 inputs get very small.

In [ ]:
# --- Locate the project root so `src/` imports work from any folder ---
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
%%writefile src/models/resnet18.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights


class ResNet18Classifier(nn.Module):
    def __init__(self, num_classes: int = 43, pretrained: bool = True, dropout: float = 0.2,
                 input_normalization: str = "imagenet", resize_to: int = None):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = resnet18(weights=weights)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))

        self.resize_to = resize_to
        self.use_norm = input_normalization == "imagenet"
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self._backbone_frozen = False

    def forward(self, x):
        if self.resize_to is not None and x.shape[-1] != self.resize_to:
            x = F.interpolate(x, size=(self.resize_to, self.resize_to),
                              mode="bilinear", align_corners=False)
        if self.use_norm:
            x = (x - self.mean) / self.std
        return self.backbone(x)

    # --- Phase control -------------------------------------------------
    def freeze_backbone(self):
        """Train only the classification head."""
        for name, p in self.named_parameters():
            p.requires_grad = name.startswith("backbone.fc")
        self._backbone_frozen = True

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True
        self._backbone_frozen = False

    def train(self, mode: bool = True):
        super().train(mode)
        # While frozen, keep BatchNorm running statistics fixed
        if mode and self._backbone_frozen:
            for m in self.backbone.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()
        return self

In [ ]:
from dataclasses import replace
import torch

from src.config import TrainConfig, get_device
from src.pipeline import (set_seed, fit, evaluate, plot_history, save_run_summary,
                          make_fake_loaders)
from src.models.resnet18 import ResNet18Classifier

cfg = TrainConfig()          # shared config: identical across all models
set_seed(cfg.seed)
device = get_device()
print("Device:", device)

In [ ]:
# ----------------------------------------------------------------------------
# DATA  --  ALEX'S PART (GTSRB download, preprocessing, train/val/test splits)
# ----------------------------------------------------------------------------
USE_FAKE_DATA = True   # <-- set to False once Alex's data pipeline is merged

if USE_FAKE_DATA:
    # Random data: only verifies the code runs. Results are meaningless.
    train_loader, val_loader, test_loader = make_fake_loaders(cfg)
else:
    # ALEX'S PART: assumed interface (see notebook 00). Adjust the call if Alex's differs.
    from src.data import get_dataloaders
    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=cfg.batch_size, num_workers=cfg.num_workers, img_size=cfg.img_size)

In [ ]:
model = ResNet18Classifier(num_classes=cfg.num_classes, pretrained=True,
                           dropout=0.2, input_normalization="imagenet", resize_to=64)
# NOTE: input_normalization="imagenet" assumes loaders output images in [0, 1] (ALEX's loaders).
#       If Alex's loaders already normalise, set input_normalization=None.
print("Parameters:", sum(p.numel() for p in model.parameters()))

### Phase 1 – train the head only

In [ ]:
model.freeze_backbone()
cfg_p1 = replace(cfg, epochs=5, lr=1e-3, scheduler="none", early_stop_patience=5)
hist_p1 = fit(model, train_loader, val_loader, cfg_p1, run_name="resnet18_phase1_head", device=device)

### Phase 2 – unfreeze and fine-tune the whole network

In [ ]:
model.unfreeze_all()
cfg_p2 = replace(cfg, lr=1e-4)      # smaller LR for the pretrained weights; other settings from shared cfg
hist_p2 = fit(model, train_loader, val_loader, cfg_p2, run_name="resnet18", device=device)
plot_history(hist_p2, title="resnet18")

In [ ]:
# Validation metrics (used for model selection). Test-set evaluation is reserved for the final experiments.
val_metrics = evaluate(model, val_loader, device, cfg.num_classes)
print({k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})
save_run_summary("resnet18", cfg_p2, val_metrics, split="val")

EVAL_ON_TEST = False   # flip only for the final results, after all tuning is done
if EVAL_ON_TEST:
    test_metrics = evaluate(model, test_loader, device, cfg.num_classes)
    save_run_summary("resnet18", cfg_p2, test_metrics, split="test")
    print({k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})

# ALEX'S PART: corrupted-image evaluation (fog, rain, blur, noise, brightness, occlusion; severity 1-5)
# is implemented by Alex. Once his corrupted test loaders exist, call evaluate(model, corrupted_loader, ...) here.